<a href="https://colab.research.google.com/github/dbh25-10120-cloud/scaling-waffle/blob/main/%EB%85%BC%EB%AC%B8%EC%9A%94%EC%95%BD1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install pymupdf transformers torch

In [13]:
pdf_path = "000002067894_20260330212044.pdf"

In [14]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

paper_text = extract_text(pdf_path)
print(paper_text[:1000])

 
저작자표시-비영리-변경금지 2.0 대한민국 
이용자는 아래의 조건을 따르는 경우에 한하여 자유롭게 
l 이 저작물을 복제, 배포, 전송, 전시, 공연 및 방송할 수 있습니다.  
다음과 같은 조건을 따라야 합니다: 
l 귀하는, 이 저작물의 재이용이나 배포의 경우, 이 저작물에 적용된 이용허락조건
을 명확하게 나타내어야 합니다.  
l 저작권자로부터 별도의 허가를 받으면 이러한 조건들은 적용되지 않습니다.  
저작권법에 따른 이용자의 권리는 위의 내용에 의하여 영향을 받지 않습니다. 
이것은 이용허락규약(Legal Code)을 이해하기 쉽게 요약한 것입니다.  
Disclaimer 
 
  
  
저작자표시. 귀하는 원저작자를 표시하여야 합니다. 
비영리. 귀하는 이 저작물을 영리 목적으로 이용할 수 없습니다. 
변경금지. 귀하는 이 저작물을 개작, 변형 또는 가공할 수 없습니다. 
 
Doctoral Thesis 
 
 
 
Hyaluronic Acid Derivatives for 
Smart Drug Delivery Systems 
 
 
 
 
 
 
 
 
 
 
 
 
 
Hyemin Kim (김 혜 민) 
Department of Materials Science and Engineering 
Pohang University of Science and Technology 
2015 
 
 
생체고분자 히알루론산을 이용한 
 
지능형 약물 전달 시스템 
 
Hyaluronic Acid Derivatives for 
Smart Drug Delivery Systems 
 
Hyaluronic Acid Derivatives for 
Smart Drug Delivery Systems 
 
by 
  
Hyemin Kim 
Department of Materials Science and Engineering (Biomaterials) 
Pohang University of Science and Technology 
  
A thesis submitted to

In [21]:
def detect_language(text, sample_size=5000):
    sample = text[:sample_size]

    total_chars = len(sample)
    korean_chars = sum(1 for c in sample if '\uac00' <= c <= '\ud7a3')

    ratio = korean_chars / total_chars

    if ratio > 0.3:
        return "ko"
    else:
        return "en"

lang = detect_language(paper_text)
print("감지된 언어:", lang)

감지된 언어: en


In [22]:
def extract_section(text, start_keywords, end_keywords):
    text_lower = text.lower()

    start_idx = -1
    for keyword in start_keywords:
        idx = text_lower.find(keyword)
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    for keyword in end_keywords:
        idx = text_lower.find(keyword, start_idx + 1)
        if idx != -1:
            end_idx = idx
            break

    return text[start_idx:end_idx].strip()


# 언어별 키워드
if lang == "ko":
    abstract = extract_section(paper_text, ["초록", "요약"], ["서론", "1. 서론"])
    conclusion = extract_section(paper_text, ["결론", "논의"], ["참고문헌"])
else:
    abstract = extract_section(paper_text, ["abstract"], ["introduction"])
    conclusion = extract_section(paper_text, ["conclusion", "discussion"], ["references"])

important_text = abstract + "\n" + conclusion

print("Abstract 길이:", len(abstract))
print("Conclusion 길이:", len(conclusion))

Abstract 길이: 693
Conclusion 길이: 55


In [23]:
from transformers import BartTokenizer, BartForConditionalGeneration

if lang == "ko":
    model_name = "digit82/kobart-summarization"
else:
    model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [24]:
import re

def split_text(text, max_length=800):
    sentences = re.split(r'(?<=[.!?다])\s+', text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) < max_length:
            current += sentence + " "
        else:
            chunks.append(current)
            current = sentence + " "

    chunks.append(current)
    return chunks

chunks = split_text(important_text)
print("chunk 개수:", len(chunks))

chunk 개수: 1


In [25]:
def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=50,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [26]:
summaries = [summarize(chunk) for chunk in chunks]
final_summary = summarize(" ".join(summaries))

print("\n===== 최종 요약 =====\n")
print(final_summary)


===== 최종 요약 =====

Non-invasive DDS via alternative routes for needle-based injection have been developed by using hyaluronic acid (HA) derivatives. Besides well-known advantages of HA including biocompatibility, biodegradability, and safety, its unique properties that can be applied to DDS have been investigated.
